[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-09-subword-tokenization.ipynb)

# Lecture 9: what your model cannot say, and subwords that can

Your A8 model has a **fixed vocabulary**: every word it saw in training, and nothing else. A
word it never saw becomes `<unk>`, and it can never produce that word in a translation, however
well it trains. This activity measures how much of *your* test set that is, then builds a
subword vocabulary that has no such gap.

When it ends you will have, in `MyDrive/CS479/assignment8`:

1. **A SentencePiece model trained on your A8 training split**, `spm.model` and `spm.vocab`.
2. **Two numbers side by side**: how many test words your word vocabulary cannot represent,
   and how many your subword vocabulary cannot. The second should be zero.
3. **A round-trip check**: text encoded to pieces and decoded back, unchanged.
4. **The three settings A9 needs**, printed for your data.

**Nothing here retrains a model.** That is Assignment 9, which reuses your A8 notebook almost
unchanged. This runs on a CPU, so no GPU is needed today.

**Running in Google Colab?** Run the next two cells. The first installs TorchLingo; the second
connects Google Drive, where your A8 files are. There is nothing to uncomment.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 9 (*Handling Morphology and Terminology in NMT*). A head start on **A9**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
# ">=0.2.3": that release trains SentencePiece quietly, instead of a line per merge.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo>=0.2.3"
import torchlingo

In [ ]:
# Connect Google Drive, where the A8 kickoff notebook wrote your split.
from torchlingo.colab import setup

env = setup(drive=True)
A8_DIR = env.drive_dir / "CS479" / "assignment8"

missing = [n for n in ("train.tsv", "val.tsv", "test.tsv") if not (A8_DIR / n).exists()]
if missing:
    raise SystemExit(
        f"Not found in {A8_DIR}: {', '.join(missing)}. These are written by Step 5 of the A8 "
        "kickoff notebook. Run that far first, or point A8_DIR at wherever your split is."
    )

import pandas as pd
from torchlingo.config import Config
from torchlingo.data_processing import NMTDataset
from torchlingo.data_processing.vocab import SentencePieceVocab
from torchlingo.diagnostics import uniform_loss
from torchlingo.preprocessing import train_sentencepiece

test = pd.read_csv(A8_DIR / "test.tsv", sep="\t", dtype=str, keep_default_na=False)
print(f"Found your A8 split in {A8_DIR}; {len(test):,} test pairs.")

## Step 1: what your word vocabulary cannot say

This rebuilds the exact vocabulary your A8 model uses (every word in the training split) and
asks how many **test** words fall outside it. Each one is an `<unk>`: on the source side the model
cannot read it, and on the target side it cannot write it, ever.

In [ ]:
words = NMTDataset(A8_DIR / "train.tsv", config=Config())   # the same vocabulary A8 built


def unknown_rate(vocab, sentences):
    """Return (unknown tokens, all tokens) for sentences encoded with vocab."""
    ids = [i for s in sentences for i in vocab.encode(s, add_special_tokens=False)]
    return sum(i == vocab.unk_idx for i in ids), len(ids)


word_unk = {}
for side, vocab in (("src", words.src_vocab), ("tgt", words.tgt_vocab)):
    unk, total = unknown_rate(vocab, test[side])
    word_unk[side] = unk
    print(f"{side}: word vocabulary {len(vocab):,};  test tokens it cannot represent: "
          f"{unk:,} of {total:,} ({100 * unk / total:.1f}%)")

print(f"\nln(V) for your target vocabulary: {uniform_loss(len(words.tgt_vocab)):.2f}")

The target-side percentage is the part to sit with. Those words are in your test references, so
every one of them is a word your model's translation is guaranteed to miss, and BLEU counts each
miss. No amount of training fixes it: the words are not in the vocabulary.

## Step 2: a subword vocabulary, trained on your training split only

SentencePiece learns a vocabulary of *pieces*: frequent words stay whole, and rare words are spelled
out of smaller pieces, down to single characters if need be. So any word made of characters it has
seen can be represented, including words it has never seen whole.

**Train it on `train.tsv` only.** Your test set must stay unseen, and a vocabulary is part of what a
model learns. Fitting it on everything would leak test material into the vocabulary, and nothing
would crash to tell you.

8,000 pieces is the size to start from: large enough for common words to stay whole, small
enough that `ln(V)` is about 9.0, which Lecture 7's slides quote. It takes about a minute.

In [ ]:
VOCAB_SIZE = 8_000

train_sentencepiece([A8_DIR / "train.tsv"], model_prefix=str(A8_DIR / "spm"), vocab_size=VOCAB_SIZE)
pieces = SentencePieceVocab(str(A8_DIR / "spm.model"))

print(f"Subword vocabulary: {len(pieces):,} pieces, shared by source and target.")
print(f"ln(V) = {uniform_loss(len(pieces)):.2f}")
print(f"Saved: {A8_DIR / 'spm.model'} and {A8_DIR / 'spm.vocab'}")

## Step 3: look at the pieces

`▁` marks the start of a word. A word that is one piece was common enough to earn its own entry;
a word split into several was rarer. Look for a word your word vocabulary called `<unk>`: here it
is spelled out of pieces instead of lost.

In [ ]:
for sentence in test["tgt"].head(3):
    as_words = words.tgt_vocab.indices_to_tokens(words.tgt_vocab.encode(sentence, add_special_tokens=False))
    as_pieces = pieces.indices_to_tokens(pieces.encode(sentence, add_special_tokens=False))
    print("WORDS ", " ".join(as_words))
    print("PIECES", " ".join(as_pieces))
    print()

## Step 4: the same count, with subwords

The same test sets, encoded with the pieces. The count should be zero. The only way it is not: a
*character* in the test set that never appears anywhere in training, such as a symbol or a letter
from another script. The cell names them if so.

In [ ]:
train = pd.read_csv(A8_DIR / "train.tsv", sep="\t", dtype=str, keep_default_na=False)

for side in ("src", "tgt"):
    unk, total = unknown_rate(pieces, test[side])
    print(f"{side}: test tokens the subword vocabulary cannot represent: {unk:,} of {total:,}   "
          f"(word vocabulary: {word_unk[side]:,})")
    if unk:
        seen = set("".join(train["src"]) + "".join(train["tgt"]))
        unseen = sorted(set("".join(test[side])) - seen)
        print(f"   characters never seen in training: {unseen}")

## Step 5: nothing is lost on the way through

Pieces are only useful if they turn back into the same text. Ten test sentences, encoded and
decoded. One subtlety: SentencePiece normalizes text first (Unicode NFKC, and runs of spaces
collapse), so the check compares against the normalized sentence. A sentence that differs before
normalization differs only in how a character was typed, not in what it says.

In [ ]:
import unicodedata


def normalized(text):
    """What SentencePiece's default normalization does: NFKC, single spaces."""
    return " ".join(unicodedata.normalize("NFKC", text).split())


sample = list(test["tgt"].head(10))
round_trip = [pieces.decode(pieces.encode(s, add_special_tokens=False)) for s in sample]
exact = sum(a == b for a, b in zip(sample, round_trip))
after_normalizing = sum(normalized(a) == b for a, b in zip(sample, round_trip))

print(f"Round trip: {exact}/10 identical; {after_normalizing}/10 identical after normalizing.")
assert after_normalizing == 10, "a sentence changed beyond normalization; ask the teaching staff"
print(f"\nexample:\n  {sample[0]}\n  {' '.join(pieces.indices_to_tokens(pieces.encode(sample[0], add_special_tokens=False)))}\n  {round_trip[0]}")

## Step 6: the price, and the three settings A9 needs

Pieces are shorter than words, so every sentence gets **longer**, measured in tokens. That costs
memory and time, and it moves one setting you have never had to think about: the **longest
translation the model is allowed to produce**. TorchLingo stops decoding at 100 tokens by default.
Under A8's word vocabulary that matched the 100-word cap. In pieces, 100 tokens is well short of
100 words, so a long translation would be cut off mid-sentence, and your A9 BLEU would drop for a
reason that has nothing to do with subwords. The cell measures your own longest sentence.

In [ ]:
word_len = train["tgt"].str.split().str.len()
piece_len = train["tgt"].map(lambda s: len(pieces.encode(s, add_special_tokens=False)))

MAX_DECODE = int(piece_len.max()) + 10    # your longest target, in pieces, with room to spare

print(f"Target sentences are {piece_len.sum() / word_len.sum():.2f}x longer in pieces than in words.")
print(f"Longest training target: {word_len.max()} words, {piece_len.max()} pieces.")
print()
print("=== For Assignment 9: change these in your A8 notebook's Step 6 ===")
print(f"  In Config(...), add:              max_decode_length={MAX_DECODE},")
print(f"  In create_dataloaders(...), add:  use_sentencepiece=True, sp_model_path=str(OUT_DIR / 'spm.model'),")
print("Everything else stays: the same files, the same split, the same training and scoring cells.")

## What A9 is, now that you have seen this

Assignment 9 retrains your A8 model with the subword vocabulary and compares the two BLEU scores.
Because it uses **the same files and the same split**, the tokenizer is the only thing that
changes, so the difference in BLEU is the tokenizer's doing. That is the whole point of the
comparison, and why nothing here re-splits your data.

Two rules for the comparison to mean anything:

- **Score the translations `translate_batch` returns.** They are ordinary text: the pieces are
  decoded back into words before you see them. Never score token ids or pieces, which are not
  comparable with A8's word-level BLEU.
- **Set `max_decode_length` as printed above**, or long translations are cut short and the
  comparison is lost to truncation.

Your A8 notebook's scoring cell already does the first; A9 needs only the three settings printed
above.